# Compare Diffusion Checkpoints With Feature Metrics

This notebook creates `answer_features.csv` with the best action extractor checkpoint, then evaluates diffusion checkpoints at steps 500, 6500, and 15000 using DINO, FVD, and action-consistency metrics.

Run this from the project container where `/workspace/...` paths are available.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch

# If the notebook is opened from a different cwd, this will still move into the repo.
REPO_ROOT = Path('/workspace/so100/so100_act_cond_diffusion')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT)
print('repo:', REPO_ROOT)

USE_POETRY = True
PYTHON = ['poetry', 'run', 'python'] if USE_POETRY else [sys.executable]

CHALLENGE_ROOT = Path('/workspace/smolvla_eval_challenge_stride5/challenge')
ANSWER_VIDEO_ROOT = Path('/workspace/smolvla_eval_challenge_stride5/answers/videos')
TRAIN_ACTION_STATS = Path('/workspace/so100_stride5/so100_action_statistics.json')

ACTION_EXTRACTOR_CKPT_DIR = Path('/workspace/so100_action_extractor/checkpoints')
ACTION_EXTRACTOR_CKPT_OVERRIDE = None  # Example: Path('/workspace/so100_action_extractor/checkpoints/epoch=...ckpt')

DIFFUSION_CKPT_DIR = Path('/workspace/so100_act_cond_diffusion_11M/checkpoints')
DIFFUSION_STEPS = [500, 6500, 15000]
EVAL_CONFIG = Path('configs/eval/smolvla_so100_eval_11M.yaml')

OUTPUT_ROOT = Path('/workspace/so100_checkpoint_feature_compare')
ANSWER_CSV = OUTPUT_ROOT / 'answer_features.csv'
SUMMARY_CSV = OUTPUT_ROOT / 'checkpoint_metric_summary.csv'

BATCH_SIZE = 4
FEATURE_BATCH_SIZE = 4
DDIM_STEPS = None  # Set an int, e.g. 50, to override the eval config.
FORCE_ANSWER_FEATURES = False
FORCE_SUBMISSION_FEATURES = False
FORCE_GENERATION = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def torch_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def scalar(value):
    if value is None:
        return None
    if hasattr(value, 'item'):
        return float(value.item())
    try:
        return float(value)
    except Exception:
        return None


def resolve_existing_path(path_text: str | Path, base_dir: Path) -> Path | None:
    path = Path(path_text)
    candidates = [path]
    if not path.is_absolute():
        candidates.append(base_dir / path)
    candidates.append(base_dir / path.name)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def find_best_action_extractor_checkpoint(ckpt_dir: Path, override: Path | None = None) -> Path:
    if override is not None:
        override = Path(override)
        if not override.exists():
            raise FileNotFoundError(override)
        return override

    ckpts = sorted(ckpt_dir.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No action extractor checkpoints found under {ckpt_dir}')

    # Lightning usually stores best_model_path / best_k_models in last.ckpt.
    last = ckpt_dir / 'last.ckpt'
    metadata_sources = [last] if last.exists() else []
    metadata_sources += [p for p in ckpts if p != last]

    best_path = None
    best_score = math.inf
    for metadata_path in metadata_sources:
        try:
            checkpoint = torch_load_checkpoint(metadata_path)
        except Exception as exc:
            print(f'[warn] could not read checkpoint metadata {metadata_path}: {exc}')
            continue
        callbacks = checkpoint.get('callbacks', {}) if isinstance(checkpoint, dict) else {}
        for state in callbacks.values():
            if not isinstance(state, dict):
                continue
            for path_text, score_value in state.get('best_k_models', {}).items():
                score = scalar(score_value)
                path = resolve_existing_path(path_text, ckpt_dir)
                if path is not None and score is not None and score < best_score:
                    best_path = path
                    best_score = score
            score = scalar(state.get('best_model_score'))
            path_text = state.get('best_model_path')
            path = resolve_existing_path(path_text, ckpt_dir) if path_text else None
            if path is not None and score is not None and score < best_score:
                best_path = path
                best_score = score

    if best_path is not None:
        print(f'best action extractor: {best_path}  val/mae={best_score:.6f}')
        return best_path

    raise RuntimeError(
        'Could not infer the best action extractor checkpoint from Lightning metadata. '
        'Set ACTION_EXTRACTOR_CKPT_OVERRIDE manually in the config cell.'
    )


def filename_step(path: Path) -> int | None:
    name = path.name
    for pattern in (r'step[=_-](\d+)', r'-(\d+)\.ckpt$', r'_(\d+)\.ckpt$'):
        match = re.search(pattern, name)
        if match:
            return int(match.group(1))
    return None


def checkpoint_global_step(path: Path) -> int | None:
    try:
        checkpoint = torch_load_checkpoint(path)
    except Exception as exc:
        print(f'[warn] could not inspect {path}: {exc}')
        return None
    step = checkpoint.get('global_step') if isinstance(checkpoint, dict) else None
    return int(step) if step is not None else None


def find_diffusion_checkpoint_for_step(ckpt_dir: Path, step: int) -> Path:
    ckpts = sorted(p for p in ckpt_dir.glob('*.ckpt') if p.name != 'last.ckpt')
    name_matches = [p for p in ckpts if filename_step(p) == step]
    if name_matches:
        return name_matches[0]

    global_step_matches = [p for p in ckpts if checkpoint_global_step(p) == step]
    if global_step_matches:
        return global_step_matches[0]

    available = [(filename_step(p), p.name) for p in ckpts]
    raise FileNotFoundError(f'No checkpoint for step={step} under {ckpt_dir}. Available filename steps: {available[:20]}')


def run(cmd: list[str], *, skip: bool = False):
    print('\n$', ' '.join(map(str, cmd)))
    if skip:
        print('[skip] output already exists')
        return
    subprocess.run([str(x) for x in cmd], cwd=REPO_ROOT, check=True)

In [ ]:
ACTION_EXTRACTOR_CKPT = find_best_action_extractor_checkpoint(
    ACTION_EXTRACTOR_CKPT_DIR,
    ACTION_EXTRACTOR_CKPT_OVERRIDE,
)

DIFFUSION_CKPTS = {
    step: find_diffusion_checkpoint_for_step(DIFFUSION_CKPT_DIR, step)
    for step in DIFFUSION_STEPS
}

print('\nDiffusion checkpoints:')
for step, path in DIFFUSION_CKPTS.items():
    print(f'  step {step}: {path}')

In [ ]:
answer_cmd = PYTHON + [
    'scripts/eval/make_answer_feature_csv.py',
    '--answer-video-root', str(ANSWER_VIDEO_ROOT),
    '--challenge-root', str(CHALLENGE_ROOT),
    '--output-csv', str(ANSWER_CSV),
    '--action-stats-path', str(TRAIN_ACTION_STATS),
    '--action-extractor-ckpt', str(ACTION_EXTRACTOR_CKPT),
    '--feature-batch-size', str(FEATURE_BATCH_SIZE),
]

run(answer_cmd, skip=ANSWER_CSV.exists() and not FORCE_ANSWER_FEATURES)
print('answer csv:', ANSWER_CSV)

In [ ]:
result_rows = []

for step, checkpoint_path in DIFFUSION_CKPTS.items():
    step_root = OUTPUT_ROOT / f'step_{step}'
    prediction_root = step_root / 'videos'
    submission_csv = step_root / 'submission_features.csv'
    details_csv = step_root / 'score_details.csv'
    summary_csv = step_root / 'score_summary.csv'
    step_root.mkdir(parents=True, exist_ok=True)

    submission_cmd = PYTHON + [
        'scripts/eval/make_submission_feature_csv.py',
        '--config', str(EVAL_CONFIG),
        '--checkpoint', str(checkpoint_path),
        '--challenge-root', str(CHALLENGE_ROOT),
        '--prediction-root', str(prediction_root),
        '--output-csv', str(submission_csv),
        '--generation-action-stats-path', str(TRAIN_ACTION_STATS),
        '--evaluator-action-stats-path', str(TRAIN_ACTION_STATS),
        '--action-extractor-ckpt', str(ACTION_EXTRACTOR_CKPT),
        '--batch-size', str(BATCH_SIZE),
        '--feature-batch-size', str(FEATURE_BATCH_SIZE),
    ]
    if DDIM_STEPS is not None:
        submission_cmd += ['--ddim-steps', str(DDIM_STEPS)]
    if FORCE_GENERATION:
        submission_cmd += ['--overwrite']

    run(submission_cmd, skip=submission_csv.exists() and not FORCE_SUBMISSION_FEATURES)

    score_cmd = PYTHON + [
        'scripts/eval/score_feature_csv.py',
        '--submission-csv', str(submission_csv),
        '--answer-csv', str(ANSWER_CSV),
        '--challenge-root', str(CHALLENGE_ROOT),
        '--action-stats-path', str(TRAIN_ACTION_STATS),
        '--details-csv', str(details_csv),
        '--summary-csv', str(summary_csv),
    ]
    run(score_cmd)

    row = pd.read_csv(summary_csv).iloc[0].to_dict()
    row['step'] = step
    row['checkpoint'] = str(checkpoint_path)
    row['submission_csv'] = str(submission_csv)
    row['details_csv'] = str(details_csv)
    result_rows.append(row)

results = pd.DataFrame(result_rows).sort_values('step')
results.to_csv(SUMMARY_CSV, index=False)
print('combined summary:', SUMMARY_CSV)

In [ ]:
metric_columns = [
    'step',
    'final_score',
    'mean_dino_component',
    'mean_fvd_component',
    'mean_action_component',
    'mean_dino_distance',
    'mean_fvd_feature_distance',
    'mean_real_action_mae',
    'mean_generated_action_mae',
    'mean_action_error_ratio',
    'checkpoint',
]

display(results[[column for column in metric_columns if column in results.columns]])